Một cơ hội mua có thể xuất hiện khi giá cổ phiếu chạm vào hoặc tiệm cận Lower Bollinger Band, đồng thời RSI ở dưới 30.

Một cơ hội bán tốt có thể xuất hiện khi giá chạm vào hoặc tiệm cận Upper Bollinger Band đồng thời RSI ở trên 70.

# Load data len (ham)

In [1]:
def loaddataMT5_FromTo(symbol, from_date, to_date, timeframe):
    import MetaTrader5 as mt5
    from datetime import datetime
    import pandas as pd 

    # Kết nối tới MetaTrader 5
    if not mt5.initialize():
        print("Khởi tạo MT5 không thành công")
        mt5.shutdown()

    from_date_str = datetime.strptime(from_date, '%Y-%m-%d')
    to_date_str = datetime.strptime(to_date, '%Y-%m-%d')
    
    # Lấy dữ liệu OHLC cho cặp tiền symbol trong khoảng thời gian đã xác định
    ohlc_data = mt5.copy_rates_range(symbol, timeframe, from_date_str, to_date_str)
    # Ngắt kết nối với MT5
    mt5.shutdown()

    # Chuyển dữ liệu nhận được thành DataFrame và hiển thị
    data = pd.DataFrame(ohlc_data)
    data['time'] = pd.to_datetime(data['time'], unit='s') # Datetime 10 chu so, neu la 13 thi parse la ms

    # ohlc_df.reset_index(inplace=True)

    data = data.rename(columns={'time': 'Datetime'})        
    data = data.rename(columns={'open': 'Open'})       
    data = data.rename(columns={'high': 'High'})       
    data = data.rename(columns={'low': 'Low'})       
    data = data.rename(columns={'close': 'Close'})       
    data = data.rename(columns={'tick_volume': 'Volume'})       

    data = pd.DataFrame(data, columns=['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume'])

    return data

In [ ]:
import MetaTrader5 as mt5
import pandas as pd
import plotly.graph_objects as go
import redis
import numpy as np
from datetime import datetime, timedelta

# ##############################################Step 0: Các tham số để lấy dữ liệu###############################
symbol = 'EURUSDs'
from_date = (datetime.now() - timedelta(days=3)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
to_date = (datetime.now() + timedelta(days=0)).strftime('%Y-%m-%d')
timeframe = mt5.TIMEFRAME_H1

# ##############################################Step 1: Lấy dữ liệu##############################################
data = loaddataMT5_FromTo(symbol, from_date, to_date, timeframe)
# Thiết lập 'Datetime' làm chỉ mục của DataFrame
data.set_index('Datetime', inplace=True)

# ##############################################Step 2: Chiến lược##############################################  
import talib
# Đặt thời gian cho SMA và Bollinger Bands
window = 20
data['SMA'] = talib.SMA(data['Close'], timeperiod=window)

# Tính toán Bollinger Bands bằng TA-Lib
data['Upper_Band'], data['Middle_Band'], data['Lower_Band'] = talib.BBANDS(
	data['Close'], timeperiod=window, nbdevup=2, nbdevdn=2, matype=0)
# Tính toán RSI bằng TA-Lib
windowRSI = 14
data['RSI'] = talib.RSI(data['Close'], timeperiod=windowRSI)

# Xác định tín hiệu mua
data['Buy_Signal'] = ((data['Close'] <= data['Lower_Band']) & (data['RSI'] < 30)) # & ((np.maximum(data['Open'], data['Close'])) >= data['Lower_Band']))
# Xác định tín hiệu bán
data['Sell_Signal'] = ((data['Close'] >= data['Upper_Band']) & (data['RSI'] > 70))

data

,Open,High,Low,Close,Volume,SMA,Upper_Band,Middle_Band,Lower_Band,RSI,Buy_Signal,Sell_Signal
Datetime,,,,,,,,,,,,
2024-11-11 17:00:00,1.06427,1.06503,1.06369,1.06476,1055,NaN,NaN,NaN,NaN,NaN,False,False
2024-11-11 18:00:00,1.06475,1.06542,1.06451,1.06520,679,NaN,NaN,NaN,NaN,NaN,False,False
2024-11-11 19:00:00,1.06520,1.06564,1.06453,1.06493,484,NaN,NaN,NaN,NaN,NaN,False,False
2024-11-11 20:00:00,1.06493,1.06517,1.06461,1.06509,354,NaN,NaN,NaN,NaN,NaN,False,False
2024-11-11 21:00:00,1.06509,1.06569,1.06495,1.06511,275,NaN,NaN,NaN,NaN,NaN,False,False
2024-11-11 22:00:00,1.06511,1.06560,1.06511,1.06558,246,NaN,NaN,NaN,NaN,NaN,False,False
2024-11-11 23:00:00,1.06558,1.06572,1.06558,1.06562,88,NaN,NaN,NaN,NaN,NaN,False,False
2024-11-12 00:00:00,1.06516,1.06568,1.06510,1.06553,182,NaN,NaN,NaN,NaN,NaN,False,False
2024-11-12 01:00:00,1.06547,1.06629,1.06543,1.06629,179,NaN,NaN,NaN,NaN,NaN,False,False


In [9]:
data.to_csv('Buoi5.2_Chienluoc Bollinger Bands, RSI.csv')

In [10]:
import pandas as pd
import plotly.graph_objects as go

####################################################################################################
# Tạo figure
fig = go.Figure()

# Thêm đường giá đóng cửa, SMA và Bollinger Bands
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Giá Đóng Cửa'))
fig.add_trace(go.Scatter(x=data.index, y=data['SMA'], mode='lines', name='SMA'))
fig.add_trace(go.Scatter(x=data.index, y=data['Upper_Band'], mode='lines', name='Bollinger Band Trên', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=data.index, y=data['Lower_Band'], mode='lines', name='Bollinger Band Dưới', line=dict(dash='dash')))

# Điền màu giữa Upper và Lower Bands
fig.add_trace(go.Scatter(x=data.index, y=data['Upper_Band'], mode='lines', name='Upper Band', line=dict(width=0)))
fig.add_trace(go.Scatter(x=data.index, y=data['Lower_Band'], mode='lines', name='Lower Band', fill='tonexty', fillcolor='rgba(204,204,204,0.2)', line=dict(width=0)))

# Vẽ các điểm cho tín hiệu mua
buy_signals = data[data['Buy_Signal']]
fig.add_trace(go.Scatter(x=buy_signals.index, y=buy_signals['Close'], mode='markers', name='Buy Signal', marker=dict(symbol='triangle-up', size=10, color='green')))

# Vẽ các điểm cho tín hiệu bán
sell_signals = data[data['Sell_Signal']]
fig.add_trace(go.Scatter(x=sell_signals.index, y=sell_signals['Close'], mode='markers', name='Sell Signal', marker=dict(symbol='triangle-down', size=10, color='red')))

# Cập nhật thông tin layout
fig.update_layout(title='Biểu Đồ Bollinger Bands và RSI', xaxis_title='Ngày', yaxis_title='Giá', showlegend=True)

# Vẽ biểu đồ
fig.show()

####################################################################################################
# Tạo figure
fig = go.Figure()

fig.add_trace(go.Scatter(x=data.index, y=data['RSI'], mode='lines', name='RSI', line=dict(color='purple')))

# Thêm các đường ngang tại các mức 70 và 30
fig.add_hline(y=70, line_dash="dash", line_color="red", line_width=0.5)
fig.add_hline(y=30, line_dash="dash", line_color="green", line_width=0.5)

# Điền màu giữa các mức 70 và 30
fig.add_trace(go.Scatter(x=data.index, y=[70]*len(data.index), mode='lines', name='RSI70', line=dict(width=0)))
fig.add_trace(go.Scatter(x=data.index, y=[30]*len(data.index), mode='lines', name='RSI30', fill='tonexty', fillcolor='rgba(204,204,204,0.2)', line=dict(width=0)))

# Cập nhật thông tin layout cho biểu đồ RSI
fig.update_layout(title='Biểu Đồ RSI', xaxis_title='Ngày', yaxis_title='RSI', showlegend=True)

# Vẽ biểu đồ
fig.show()